In [1]:
#Importing the Libraries

!pip install simpletransformers
# !pip install transformers-interpret

import pandas
import numpy
from sklearn.metrics import f1_score, classification_report
from simpletransformers.classification import ClassificationModel
# from transformers_interpret import SequenceClassificationExplainer

In [2]:
trainPath = "../input/normalizedgoemotions/GoE-Nov17/newTrainData.csv"
testPath = "../input/normalizedgoemotions/GoE-Nov17/newTestData.csv"

emoList = ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 
'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism',
'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']

In [3]:
train_df = pandas.read_csv(trainPath)
train_df = train_df.sample(frac=1).reset_index(drop=True)

newCol = []
for label in train_df['label']:
    newCol.append(emoList.index(label))
train_df['label_desc'] = train_df['label']
train_df['label'] = newCol
train_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

train_df.head()

In [4]:
test_df = pandas.read_csv(testPath)
test_df = test_df.sample(frac=1).reset_index(drop=True)

newCol = []
for label in test_df['label']:
    newCol.append(emoList.index(label))
test_df['label_desc'] = test_df['label']
test_df['label'] = newCol
test_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

test_df.head()

In [5]:
# Training Roberta Transformer on GoEmotions as MultiLabel Classification

model = ClassificationModel('roberta', 'roberta-base', num_labels=len(emoList), 
                                      args={'train_batch_size':4, 'fp16': False, 'overwrite_output_dir': True,                                         
                                            'gradient_accumulation_steps':8, 'learning_rate': 3e-5,
                                            'num_train_epochs': 1, "reprocess_input_data": True, 
                                            'max_seq_length': 128})

model.train_model(train_df)
result, model_outputs, wrong_predictions = model.eval_model(test_df)
predictions, raw_outputs = model.predict(list(test_df['text']))

print('-'*100)

print(f"Accuracy & Loss - {result}")
print(f"F1-Micro Score - {f1_score(list(test_df['labels']), predictions, average='micro')}")
print(f"F1-Macro Score - {f1_score(list(test_df['labels']), predictions, average='macro')}")

In [6]:
print(classification_report(test_df['labels'], predictions, target_names=emoList, digits=len(emoList)))

In [8]:
import shap
from transformers import TextClassificationPipeline

pipe = TextClassificationPipeline(model=model.model.cpu(), tokenizer=model.tokenizer, return_all_scores=True)

def score_and_visualize(text):
    prediction = pipe([text])
    print(prediction[0])

    explainer = shap.Explainer(pipe)
    shap_values = explainer([text])

    shap.plots.text(shap_values)
    
score_and_visualize('After many years of training, I finally succeeded in achieving '
                    'my black belt in machine learning. The road was very rough but '
                    'I could not be more happy with the result!')

In [ ]:
# idx = 12
# cls_explainer = SequenceClassificationExplainer(model.model, model.tokenizer, custom_labels=emoList)
# word_attributions = cls_explainer(test_df.iloc[idx]['text'])
# label = emoList[int(cls_explainer.predicted_class_index)]
# word_attributions

# print(f"Text : {test_df.iloc[idx]['text']}")
# print('-'*100)
# print(f"True Label : {test_df.iloc[idx]['label_desc']}")
# print(f"Predicted Label : {label}")